In [ ]:
# ============================================================
# PT -> PA Cross-Physics Mapping
# Galerkin Neural Operator / Galerkin Transformer
#
# Input:
#     PT [B, 1, 501, 200]
#
# Output:
#     PA [B, 1, 501, 200]
#
# Architecture:
#
# PT field
#   ->
# Patch embedding
#   ->
# Coordinate embedding
#   ->
# Galerkin Operator Blocks
#   ->
# Field decoder
#   ->
# PA field
#
# Galerkin attention:
#
# Y = Q (K^T V / N)
#
# No softmax attention.
# ============================================================


# ============================================================
# Imports
# ============================================================

import os
import csv
import glob
import random

from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import h5py

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader


# ============================================================
# Config
# ============================================================

DATA_ROOT = "Training dataset"


RUN_TIME = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUT_DIR = os.path.join(
    "galerkin_operator_results",
    f"run_{RUN_TIME}"
)


os.makedirs(
    OUT_DIR,
    exist_ok=True
)


print(
    "Results will be saved to:"
)

print(
    OUT_DIR
)


# ============================================================
# Dataset split
# ============================================================

TRAIN_RATIO = 0.8

VAL_RATIO = 0.1

TEST_RATIO = 0.1


SEED = 42


# ============================================================
# Device
# ============================================================

DEVICE = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"
)


print(
    "Using device:",
    DEVICE
)


# ============================================================
# Data dimensions
#
# T = 501
# X = 200
# ============================================================

NT = 501

NX = 200


# ============================================================
# Training parameters
# ============================================================

EPOCHS = 1000

BATCH_SIZE = 4

LR = 1e-3

WEIGHT_DECAY = 1e-5


# ============================================================
# Galerkin Operator parameters
# ============================================================

IN_CHANNELS = 1

OUT_CHANNELS = 1


# ------------------------------------------------------------
# Patch size
#
# 501 will first be padded to 504.
#
# 504 / 4 = 126
# 200 / 4 = 50
#
# Number of tokens:
#
# 126 x 50 = 6300
# ------------------------------------------------------------

PATCH_T = 4

PATCH_X = 4


# ------------------------------------------------------------
# Embedding dimension
# ------------------------------------------------------------

D_MODEL = 96


# ------------------------------------------------------------
# Multi-head Galerkin attention
# ------------------------------------------------------------

N_HEADS = 4


# ------------------------------------------------------------
# Number of operator blocks
# ------------------------------------------------------------

N_LAYERS = 4


# ------------------------------------------------------------
# Feed-forward expansion
# ------------------------------------------------------------

MLP_RATIO = 2


# ------------------------------------------------------------
# Dropout
# ------------------------------------------------------------

DROPOUT = 0.0


# ============================================================
# Random seed
# ============================================================

random.seed(
    SEED
)

np.random.seed(
    SEED
)

torch.manual_seed(
    SEED
)


if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        SEED
    )


# ============================================================
# Find all MAT files
# ============================================================

all_files = glob.glob(

    os.path.join(
        DATA_ROOT,
        "**",
        "*.mat"
    ),

    recursive=True
)


all_files = sorted(
    all_files
)


print(
    "Total .mat samples found:",
    len(all_files)
)


assert len(all_files) > 0, \
    f"No .mat files found under: {DATA_ROOT}"


if len(all_files) != 1000:

    print(

        f"Warning: expected 1000 samples, "
        f"but found {len(all_files)}"
    )


# ============================================================
# Random split
# ============================================================

random.shuffle(
    all_files
)


n_total = len(
    all_files
)


n_train = int(
    n_total * TRAIN_RATIO
)


n_val = int(
    n_total * VAL_RATIO
)


train_files = all_files[
    :n_train
]


val_files = all_files[
    n_train:
    n_train + n_val
]


test_files = all_files[
    n_train + n_val:
]


print(
    "\nDataset split:"
)


print(
    "Train samples:",
    len(train_files)
)


print(
    "Val samples  :",
    len(val_files)
)


print(
    "Test samples :",
    len(test_files)
)


# ============================================================
# Compute normalization statistics
#
# IMPORTANT:
# training data only
# ============================================================

def compute_statistics(
    file_list
):

    pt_sum = 0.0

    pt_sq_sum = 0.0


    pa_sum = 0.0

    pa_sq_sum = 0.0


    n_elements = 0


    print(
        "\nCalculating normalization statistics..."
    )


    for mat_path in tqdm(

        file_list,

        desc="Statistics",

        ncols=120
    ):


        with h5py.File(
            mat_path,
            "r"
        ) as f:


            PT = np.asarray(

                f["PT"],

                dtype=np.float64
            )


            PA = np.asarray(

                f["PA"],

                dtype=np.float64
            )


        # ====================================================
        # Shape
        # ====================================================

        assert PT.shape == (NT, NX), \
            f"Wrong PT shape in {mat_path}: {PT.shape}"


        assert PA.shape == (NT, NX), \
            f"Wrong PA shape in {mat_path}: {PA.shape}"


        pt_sum += PT.sum()


        pt_sq_sum += np.square(
            PT
        ).sum()


        pa_sum += PA.sum()


        pa_sq_sum += np.square(
            PA
        ).sum()


        n_elements += PT.size


    # ========================================================
    # Mean
    # ========================================================

    pt_mean = (
        pt_sum
        /
        n_elements
    )


    pa_mean = (
        pa_sum
        /
        n_elements
    )


    # ========================================================
    # Variance
    # ========================================================

    pt_var = (

        pt_sq_sum
        /
        n_elements

        -

        pt_mean ** 2
    )


    pa_var = (

        pa_sq_sum
        /
        n_elements

        -

        pa_mean ** 2
    )


    pt_var = max(
        pt_var,
        0.0
    )


    pa_var = max(
        pa_var,
        0.0
    )


    # ========================================================
    # Standard deviation
    # ========================================================

    pt_std = (
        np.sqrt(
            pt_var
        )
        +
        1e-8
    )


    pa_std = (
        np.sqrt(
            pa_var
        )
        +
        1e-8
    )


    return (

        pt_mean,
        pt_std,

        pa_mean,
        pa_std
    )


# ============================================================
# Calculate statistics
# ============================================================

(
    pt_mean,
    pt_std,

    pa_mean,
    pa_std

) = compute_statistics(
    train_files
)


print(
    "\nNormalization statistics:"
)


print(
    f"PT mean = {pt_mean:.6e}"
)


print(
    f"PT std  = {pt_std:.6e}"
)


print(
    f"PA mean = {pa_mean:.6e}"
)


print(
    f"PA std  = {pa_std:.6e}"
)


# ============================================================
# Save normalization
# ============================================================

np.savez(

    os.path.join(

        OUT_DIR,

        "normalization_parameters.npz"
    ),

    pt_mean=pt_mean,

    pt_std=pt_std,

    pa_mean=pa_mean,

    pa_std=pa_std
)


# ============================================================
# Dataset
# ============================================================

class MatHeatAcousticDataset(Dataset):


    def __init__(
        self,
        file_list,
        pt_mean,
        pt_std,
        pa_mean,
        pa_std
    ):


        self.file_list = file_list


        self.pt_mean = pt_mean

        self.pt_std = pt_std


        self.pa_mean = pa_mean

        self.pa_std = pa_std


    # ========================================================
    # Length
    # ========================================================

    def __len__(
        self
    ):

        return len(
            self.file_list
        )


    # ========================================================
    # Get item
    # ========================================================

    def __getitem__(
        self,
        idx
    ):


        mat_path = self.file_list[
            idx
        ]


        with h5py.File(
            mat_path,
            "r"
        ) as f:


            PT = np.asarray(

                f["PT"],

                dtype=np.float32
            )


            PA = np.asarray(

                f["PA"],

                dtype=np.float32
            )


        # ====================================================
        # Shape
        # ====================================================

        assert PT.shape == (NT, NX), \
            f"Wrong PT shape in {mat_path}: {PT.shape}"


        assert PA.shape == (NT, NX), \
            f"Wrong PA shape in {mat_path}: {PA.shape}"


        # ====================================================
        # Normalize
        # ====================================================

        PT = (

            PT
            -
            self.pt_mean

        ) / self.pt_std


        PA = (

            PA
            -
            self.pa_mean

        ) / self.pa_std


        # ====================================================
        # [T,X]
        #
        # ->
        #
        # [C,T,X]
        # ====================================================

        PT = torch.tensor(

            PT,

            dtype=torch.float32

        ).unsqueeze(
            0
        )


        PA = torch.tensor(

            PA,

            dtype=torch.float32

        ).unsqueeze(
            0
        )


        return (
            PT,
            PA
        )


    # ========================================================
    # Denormalize PA
    # ========================================================

    def denormalize_pa(
        self,
        x
    ):


        return (

            x
            *
            self.pa_std

            +

            self.pa_mean
        )


    # ========================================================
    # Denormalize PT
    # ========================================================

    def denormalize_pt(
        self,
        x
    ):


        return (

            x
            *
            self.pt_std

            +

            self.pt_mean
        )


# ============================================================
# Datasets
# ============================================================

train_dataset = MatHeatAcousticDataset(

    train_files,

    pt_mean,
    pt_std,

    pa_mean,
    pa_std
)


val_dataset = MatHeatAcousticDataset(

    val_files,

    pt_mean,
    pt_std,

    pa_mean,
    pa_std
)


test_dataset = MatHeatAcousticDataset(

    test_files,

    pt_mean,
    pt_std,

    pa_mean,
    pa_std
)


# ============================================================
# DataLoaders
# ============================================================

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


val_loader = DataLoader(

    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


# ============================================================
# Dataset check
# ============================================================

PT_batch, PA_batch = next(

    iter(
        train_loader
    )
)


print(
    "\nBatch check:"
)


print(
    "PT batch shape:",
    PT_batch.shape
)


print(
    "PA batch shape:",
    PA_batch.shape
)


# ============================================================
# Patch Embedding
#
# [B,1,504,200]
#
# ->
#
# [B,D_MODEL,126,50]
#
# ============================================================

class PatchEmbedding(nn.Module):


    def __init__(
        self,
        in_channels,
        embed_dim,
        patch_t,
        patch_x
    ):


        super().__init__()


        self.projection = nn.Conv2d(

            in_channels,

            embed_dim,

            kernel_size=(
                patch_t,
                patch_x
            ),

            stride=(
                patch_t,
                patch_x
            )
        )


    def forward(
        self,
        x
    ):


        return self.projection(
            x
        )


# ============================================================
# Coordinate embedding
#
# Galerkin operator should know where each token is located.
#
# Coordinates:
#
# (t, x)
#
# normalized to [-1,1].
# ============================================================

class CoordinateEmbedding(nn.Module):


    def __init__(
        self,
        embed_dim
    ):


        super().__init__()


        self.net = nn.Sequential(

            nn.Linear(
                2,
                embed_dim
            ),

            nn.GELU(),

            nn.Linear(
                embed_dim,
                embed_dim
            )
        )


    def forward(
        self,
        batch_size,
        height,
        width,
        device
    ):


        # ====================================================
        # t coordinate
        # ====================================================

        t = torch.linspace(

            -1.0,

            1.0,

            height,

            device=device
        )


        # ====================================================
        # x coordinate
        # ====================================================

        x = torch.linspace(

            -1.0,

            1.0,

            width,

            device=device
        )


        tt, xx = torch.meshgrid(

            t,

            x,

            indexing="ij"
        )


        coords = torch.stack(

            [
                tt,
                xx
            ],

            dim=-1
        )


        # ====================================================
        # [H,W,2]
        #
        # ->
        #
        # [N,2]
        # ====================================================

        coords = coords.reshape(

            -1,

            2
        )


        coords = self.net(
            coords
        )


        # ====================================================
        # [1,N,C]
        #
        # ->
        #
        # [B,N,C]
        # ====================================================

        coords = coords.unsqueeze(
            0
        )


        coords = coords.expand(

            batch_size,

            -1,

            -1
        )


        return coords


# ============================================================
# Galerkin Attention
#
# Standard Transformer:
#
# softmax(Q K^T) V
#
#
# Galerkin-style linear attention:
#
# Q (K^T V / N)
#
#
# Complexity:
#
# avoids explicitly constructing N x N attention matrix.
# ============================================================

class GalerkinAttention(nn.Module):


    def __init__(
        self,
        dim,
        num_heads
    ):


        super().__init__()


        assert (
            dim % num_heads == 0
        ), \
            "D_MODEL must be divisible by N_HEADS"


        self.dim = dim

        self.num_heads = num_heads

        self.head_dim = (
            dim
            //
            num_heads
        )


        # ====================================================
        # Q K V projections
        # ====================================================

        self.q_proj = nn.Linear(
            dim,
            dim,
            bias=False
        )


        self.k_proj = nn.Linear(
            dim,
            dim,
            bias=False
        )


        self.v_proj = nn.Linear(
            dim,
            dim,
            bias=False
        )


        # ====================================================
        # Output projection
        # ====================================================

        self.out_proj = nn.Linear(
            dim,
            dim
        )


        # ====================================================
        # Normalization used before Galerkin contraction
        # ====================================================

        self.q_norm = nn.LayerNorm(
            self.head_dim
        )


        self.k_norm = nn.LayerNorm(
            self.head_dim
        )


        self.v_norm = nn.LayerNorm(
            self.head_dim
        )


    def forward(
        self,
        x
    ):


        # ====================================================
        # Input:
        #
        # x [B,N,C]
        # ====================================================

        B, N, C = x.shape


        # ====================================================
        # Q K V
        # ====================================================

        q = self.q_proj(
            x
        )


        k = self.k_proj(
            x
        )


        v = self.v_proj(
            x
        )


        # ====================================================
        # [B,N,C]
        #
        # ->
        #
        # [B,N,H,D]
        # ====================================================

        q = q.reshape(

            B,
            N,
            self.num_heads,
            self.head_dim
        )


        k = k.reshape(

            B,
            N,
            self.num_heads,
            self.head_dim
        )


        v = v.reshape(

            B,
            N,
            self.num_heads,
            self.head_dim
        )


        # ====================================================
        # Normalize features
        # ====================================================

        q = self.q_norm(
            q
        )


        k = self.k_norm(
            k
        )


        v = self.v_norm(
            v
        )


        # ====================================================
        # [B,N,H,D]
        #
        # ->
        #
        # [B,H,N,D]
        # ====================================================

        q = q.permute(
            0,
            2,
            1,
            3
        )


        k = k.permute(
            0,
            2,
            1,
            3
        )


        v = v.permute(
            0,
            2,
            1,
            3
        )


        # ====================================================
        # Galerkin projection
        #
        # K^T V
        #
        # [B,H,D,N]
        #
        # x
        #
        # [B,H,N,D]
        #
        # ->
        #
        # [B,H,D,D]
        #
        # IMPORTANT:
        #
        # We never create [N,N].
        # ====================================================

        kv = torch.einsum(

            "bhnd,bhne->bhde",

            k,

            v
        )


        # ====================================================
        # Integral / quadrature normalization
        # ====================================================

        kv = (
            kv
            /
            float(N)
        )


        # ====================================================
        # Q (K^T V)
        #
        # [B,H,N,D]
        #
        # x
        #
        # [B,H,D,D]
        #
        # ->
        #
        # [B,H,N,D]
        # ====================================================

        out = torch.einsum(

            "bhnd,bhde->bhne",

            q,

            kv
        )


        # ====================================================
        # Merge heads
        # ====================================================

        out = out.permute(

            0,
            2,
            1,
            3

        ).contiguous()


        out = out.reshape(

            B,
            N,
            C
        )


        # ====================================================
        # Output projection
        # ====================================================

        out = self.out_proj(
            out
        )


        return out


# ============================================================
# Feed Forward Network
# ============================================================

class FeedForward(nn.Module):


    def __init__(
        self,
        dim,
        mlp_ratio=2,
        dropout=0.0
    ):


        super().__init__()


        hidden_dim = int(

            dim
            *
            mlp_ratio
        )


        self.net = nn.Sequential(

            nn.Linear(
                dim,
                hidden_dim
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                hidden_dim,
                dim
            ),

            nn.Dropout(
                dropout
            )
        )


    def forward(
        self,
        x
    ):


        return self.net(
            x
        )


# ============================================================
# Galerkin Operator Block
# ============================================================

class GalerkinBlock(nn.Module):


    def __init__(
        self,
        dim,
        num_heads,
        mlp_ratio=2,
        dropout=0.0
    ):


        super().__init__()


        # ====================================================
        # Pre-normalization
        # ====================================================

        self.norm1 = nn.LayerNorm(
            dim
        )


        self.attention = GalerkinAttention(

            dim=dim,

            num_heads=num_heads
        )


        self.norm2 = nn.LayerNorm(
            dim
        )


        self.ffn = FeedForward(

            dim=dim,

            mlp_ratio=mlp_ratio,

            dropout=dropout
        )


    def forward(
        self,
        x
    ):


        # ====================================================
        # Galerkin operator
        # ====================================================

        x = (

            x

            +

            self.attention(

                self.norm1(
                    x
                )
            )
        )


        # ====================================================
        # Local nonlinear channel mixing
        # ====================================================

        x = (

            x

            +

            self.ffn(

                self.norm2(
                    x
                )
            )
        )


        return x


# ============================================================
# Local refinement block
#
# Galerkin attention captures global interactions.
#
# Convolution restores local spatial-temporal structure.
# ============================================================

class LocalRefinement(nn.Module):


    def __init__(
        self,
        channels
    ):


        super().__init__()


        self.block = nn.Sequential(

            nn.Conv2d(

                channels,

                channels,

                kernel_size=3,

                padding=1,

                bias=False
            ),

            nn.BatchNorm2d(
                channels
            ),

            nn.GELU(),


            nn.Conv2d(

                channels,

                channels,

                kernel_size=3,

                padding=1,

                bias=False
            ),

            nn.BatchNorm2d(
                channels
            )
        )


    def forward(
        self,
        x
    ):


        return F.gelu(

            x
            +
            self.block(
                x
            )
        )


# ============================================================
# Galerkin Neural Operator
# ============================================================

class GalerkinOperator2D(nn.Module):


    def __init__(
        self,
        in_channels=1,
        out_channels=1,
        embed_dim=96,
        num_heads=4,
        num_layers=4,
        patch_t=4,
        patch_x=4,
        mlp_ratio=2,
        dropout=0.0
    ):


        super().__init__()


        self.patch_t = patch_t

        self.patch_x = patch_x


        # ====================================================
        # Patch embedding
        # ====================================================

        self.patch_embedding = PatchEmbedding(

            in_channels=in_channels,

            embed_dim=embed_dim,

            patch_t=patch_t,

            patch_x=patch_x
        )


        # ====================================================
        # Coordinate embedding
        # ====================================================

        self.coordinate_embedding = CoordinateEmbedding(

            embed_dim=embed_dim
        )


        # ====================================================
        # Operator blocks
        # ====================================================

        self.blocks = nn.ModuleList(

            [

                GalerkinBlock(

                    dim=embed_dim,

                    num_heads=num_heads,

                    mlp_ratio=mlp_ratio,

                    dropout=dropout
                )

                for _ in range(
                    num_layers
                )
            ]
        )


        # ====================================================
        # Final token normalization
        # ====================================================

        self.norm = nn.LayerNorm(
            embed_dim
        )


        # ====================================================
        # Local refinement on latent field
        # ====================================================

        self.local_refine = LocalRefinement(

            embed_dim
        )


        # ====================================================
        # Decoder
        #
        # latent patch field
        #
        # ->
        #
        # full-resolution PA
        # ====================================================

        self.decoder = nn.Sequential(

            nn.ConvTranspose2d(

                embed_dim,

                64,

                kernel_size=(
                    patch_t,
                    patch_x
                ),

                stride=(
                    patch_t,
                    patch_x
                )
            ),

            nn.GELU(),


            nn.Conv2d(

                64,

                32,

                kernel_size=3,

                padding=1
            ),

            nn.GELU(),


            nn.Conv2d(

                32,

                out_channels,

                kernel_size=3,

                padding=1
            )
        )


    # ========================================================
    # Padding
    # ========================================================

    def pad_input(
        self,
        x
    ):


        T = x.shape[
            -2
        ]


        X = x.shape[
            -1
        ]


        # ====================================================
        # Find next multiple of patch size
        # ====================================================

        pad_t = (

            self.patch_t
            -
            T % self.patch_t

        ) % self.patch_t


        pad_x = (

            self.patch_x
            -
            X % self.patch_x

        ) % self.patch_x


        # ====================================================
        # Pad:
        #
        # (left, right, top, bottom)
        # ====================================================

        x = F.pad(

            x,

            (
                0,
                pad_x,
                0,
                pad_t
            ),

            mode="replicate"
        )


        return (

            x,
            pad_t,
            pad_x
        )


    # ========================================================
    # Forward
    # ========================================================

    def forward(
        self,
        x
    ):


        # ====================================================
        # Original size
        # ====================================================

        B = x.shape[
            0
        ]


        original_t = x.shape[
            -2
        ]


        original_x = x.shape[
            -1
        ]


        # ====================================================
        # Padding
        #
        # 501 x 200
        #
        # ->
        #
        # 504 x 200
        # ====================================================

        x, _, _ = self.pad_input(
            x
        )


        # ====================================================
        # Patch embedding
        #
        # [B,1,504,200]
        #
        # ->
        #
        # [B,C,126,50]
        # ====================================================

        x = self.patch_embedding(
            x
        )


        B, C, H, W = x.shape


        # ====================================================
        # Convert field into tokens
        #
        # [B,C,H,W]
        #
        # ->
        #
        # [B,N,C]
        #
        # N = H * W
        # ====================================================

        tokens = x.flatten(
            2
        )


        tokens = tokens.transpose(
            1,
            2
        )


        # ====================================================
        # Coordinate information
        # ====================================================

        coordinates = self.coordinate_embedding(

            batch_size=B,

            height=H,

            width=W,

            device=x.device
        )


        tokens = (
            tokens
            +
            coordinates
        )


        # ====================================================
        # Galerkin operator layers
        # ====================================================

        for block in self.blocks:

            tokens = block(
                tokens
            )


        # ====================================================
        # Final normalization
        # ====================================================

        tokens = self.norm(
            tokens
        )


        # ====================================================
        # Tokens -> latent field
        #
        # [B,N,C]
        #
        # ->
        #
        # [B,C,H,W]
        # ====================================================

        x = tokens.transpose(
            1,
            2
        )


        x = x.reshape(

            B,

            C,

            H,

            W
        )


        # ====================================================
        # Local refinement
        # ====================================================

        x = self.local_refine(
            x
        )


        # ====================================================
        # Decode
        #
        # [B,C,126,50]
        #
        # ->
        #
        # [B,1,504,200]
        # ====================================================

        x = self.decoder(
            x
        )


        # ====================================================
        # Crop back to original size
        #
        # ->
        #
        # [B,1,501,200]
        # ====================================================

        x = x[
            :,
            :,
            :original_t,
            :original_x
        ]


        return x


# ============================================================
# Build model
# ============================================================

model = GalerkinOperator2D(

    in_channels=IN_CHANNELS,

    out_channels=OUT_CHANNELS,

    embed_dim=D_MODEL,

    num_heads=N_HEADS,

    num_layers=N_LAYERS,

    patch_t=PATCH_T,

    patch_x=PATCH_X,

    mlp_ratio=MLP_RATIO,

    dropout=DROPOUT

).to(
    DEVICE
)


# ============================================================
# Shape check
# ============================================================

print(
    "\nChecking model dimensions..."
)


with torch.no_grad():


    test_input = torch.randn(

        1,

        1,

        NT,

        NX,

        device=DEVICE
    )


    test_output = model(
        test_input
    )


print(
    "Input shape :",
    test_input.shape
)


print(
    "Output shape:",
    test_output.shape
)


assert (

    test_output.shape

    ==

    test_input.shape

), \
    f"Output shape mismatch: {test_output.shape}"


del test_input

del test_output


if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ============================================================
# Parameters
# ============================================================

n_params = sum(

    p.numel()

    for p in model.parameters()

    if p.requires_grad
)


print(

    f"\nTrainable parameters: "
    f"{n_params:,}"
)


# ============================================================
# Relative L2
# ============================================================

def relative_l2(
    pred,
    target
):


    numerator = torch.norm(

        pred
        -
        target
    )


    denominator = (

        torch.norm(
            target
        )

        +

        1e-8
    )


    return (

        numerator
        /
        denominator
    )


# ============================================================
# Train one epoch
# ============================================================

def train_one_epoch(
    model,
    loader,
    optimizer
):


    model.train()


    total_mse = 0.0

    total_rel = 0.0


    for heat, acoustic in loader:


        heat = heat.to(

            DEVICE,

            non_blocking=True
        )


        acoustic = acoustic.to(

            DEVICE,

            non_blocking=True
        )


        # ====================================================
        # Forward
        # ====================================================

        pred = model(
            heat
        )


        # ====================================================
        # MSE
        # ====================================================

        mse = F.mse_loss(

            pred,

            acoustic
        )


        # ====================================================
        # Relative L2
        # ====================================================

        rel = relative_l2(

            pred,

            acoustic
        )


        # ====================================================
        # Same loss as previous models
        # ====================================================

        loss = (

            mse

            +

            0.1
            *
            rel
        )


        # ====================================================
        # Backprop
        # ====================================================

        optimizer.zero_grad(
            set_to_none=True
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(

            model.parameters(),

            max_norm=1.0
        )


        optimizer.step()


        total_mse += mse.item()

        total_rel += rel.item()


    return (

        total_mse
        /
        len(loader),

        total_rel
        /
        len(loader)
    )


# ============================================================
# Evaluate
# ============================================================

@torch.no_grad()
def evaluate(
    model,
    loader
):


    model.eval()


    total_mse = 0.0

    total_mae = 0.0

    total_rel = 0.0


    for heat, acoustic in loader:


        heat = heat.to(

            DEVICE,

            non_blocking=True
        )


        acoustic = acoustic.to(

            DEVICE,

            non_blocking=True
        )


        # ====================================================
        # Prediction
        # ====================================================

        pred = model(
            heat
        )


        # ====================================================
        # Metrics
        # ====================================================

        mse = F.mse_loss(

            pred,

            acoustic
        )


        mae = F.l1_loss(

            pred,

            acoustic
        )


        rel = relative_l2(

            pred,

            acoustic
        )


        total_mse += mse.item()

        total_mae += mae.item()

        total_rel += rel.item()


    return (

        total_mse
        /
        len(loader),

        total_mae
        /
        len(loader),

        total_rel
        /
        len(loader)
    )


# ============================================================
# Plot training curves
# ============================================================

def plot_loss(
    log_path
):


    data = np.loadtxt(

        log_path,

        delimiter=",",

        skiprows=1
    )


    if data.ndim == 1:

        data = data[
            None,
            :
        ]


    epoch = data[
        :,
        0
    ]


    train_mse = data[
        :,
        1
    ]


    train_rel = data[
        :,
        2
    ]


    val_mse = data[
        :,
        3
    ]


    val_rel = data[
        :,
        5
    ]


    # ========================================================
    # MSE
    # ========================================================

    plt.figure(
        figsize=(5, 3.8)
    )


    plt.semilogy(

        epoch,

        train_mse,

        label="Train MSE"
    )


    plt.semilogy(

        epoch,

        val_mse,

        label="Validation MSE"
    )


    plt.xlabel(
        "Epoch"
    )


    plt.ylabel(
        "MSE"
    )


    plt.legend(
        frameon=False
    )


    plt.tight_layout()


    plt.savefig(

        os.path.join(
            OUT_DIR,
            "loss_curve.png"
        ),

        dpi=300
    )


    plt.close()


    # ========================================================
    # Relative L2
    # ========================================================

    plt.figure(
        figsize=(5, 3.8)
    )


    plt.semilogy(

        epoch,

        train_rel,

        label="Train Rel. L2"
    )


    plt.semilogy(

        epoch,

        val_rel,

        label="Validation Rel. L2"
    )


    plt.xlabel(
        "Epoch"
    )


    plt.ylabel(
        "Relative L2"
    )


    plt.legend(
        frameon=False
    )


    plt.tight_layout()


    plt.savefig(

        os.path.join(
            OUT_DIR,
            "relative_l2_curve.png"
        ),

        dpi=300
    )


    plt.close()


# ============================================================
# Plot prediction
# ============================================================

@torch.no_grad()
def plot_prediction(
    model,
    dataset,
    sample_index=0
):


    model.eval()


    PT, PA = dataset[
        sample_index
    ]


    # ========================================================
    # Predict
    # ========================================================

    PT_gpu = PT.unsqueeze(
        0
    ).to(
        DEVICE
    )


    pred = model(
        PT_gpu
    )


    pred = (

        pred
        .cpu()
        .squeeze(0)
        .squeeze(0)
        .numpy()
    )


    PT = (

        PT
        .squeeze(0)
        .numpy()
    )


    PA = (

        PA
        .squeeze(0)
        .numpy()
    )


    # ========================================================
    # Denormalization
    # ========================================================

    PT_real = dataset.denormalize_pt(
        PT
    )


    PA_real = dataset.denormalize_pa(
        PA
    )


    pred_real = dataset.denormalize_pa(
        pred
    )


    # ========================================================
    # Error
    # ========================================================

    err = (

        pred_real
        -
        PA_real
    )


    # ========================================================
    # Visualization scale
    # ========================================================

    vmax = np.max(

        np.abs(
            PA_real
        )
    )


    vmax = max(
        vmax,
        1e-12
    )


    evmax = np.max(

        np.abs(
            err
        )
    )


    evmax = max(
        evmax,
        1e-12
    )


    # ========================================================
    # Figure
    # ========================================================

    plt.figure(
        figsize=(12, 3)
    )


    # --------------------------------------------------------
    # PT
    # --------------------------------------------------------

    plt.subplot(
        1,
        4,
        1
    )


    plt.imshow(

        PT_real,

        aspect="auto",

        cmap="inferno"
    )


    plt.title(
        "Input PT"
    )


    plt.xlabel(
        "x"
    )


    plt.ylabel(
        "t"
    )


    plt.colorbar()


    # --------------------------------------------------------
    # GT PA
    # --------------------------------------------------------

    plt.subplot(
        1,
        4,
        2
    )


    plt.imshow(

        PA_real,

        aspect="auto",

        cmap="seismic",

        vmin=-vmax,

        vmax=vmax
    )


    plt.title(
        "GT PA"
    )


    plt.xlabel(
        "x"
    )


    plt.ylabel(
        "t"
    )


    plt.colorbar()


    # --------------------------------------------------------
    # Predicted PA
    # --------------------------------------------------------

    plt.subplot(
        1,
        4,
        3
    )


    plt.imshow(

        pred_real,

        aspect="auto",

        cmap="seismic",

        vmin=-vmax,

        vmax=vmax
    )


    plt.title(
        "Pred PA"
    )


    plt.xlabel(
        "x"
    )


    plt.ylabel(
        "t"
    )


    plt.colorbar()


    # --------------------------------------------------------
    # Error
    # --------------------------------------------------------

    plt.subplot(
        1,
        4,
        4
    )


    plt.imshow(

        err,

        aspect="auto",

        cmap="seismic",

        vmin=-evmax,

        vmax=evmax
    )


    plt.title(
        "Error"
    )


    plt.xlabel(
        "x"
    )


    plt.ylabel(
        "t"
    )


    plt.colorbar()


    plt.tight_layout()


    plt.savefig(

        os.path.join(
            OUT_DIR,
            "prediction_comparison.png"
        ),

        dpi=300,

        bbox_inches="tight"
    )


    plt.close()


    # ========================================================
    # Save numerical result
    # ========================================================

    np.savez(

        os.path.join(
            OUT_DIR,
            "prediction_result.npz"
        ),

        PT=PT_real,

        PA=PA_real,

        PA_pred=pred_real,

        error=err
    )


# ============================================================
# Optimizer
# ============================================================

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LR,

    weight_decay=WEIGHT_DECAY
)


# ============================================================
# Learning rate scheduler
# ============================================================

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(

    optimizer,

    T_max=EPOCHS
)


# ============================================================
# Training log
# ============================================================

log_path = os.path.join(

    OUT_DIR,

    "training_log.csv"
)


with open(

    log_path,

    "w",

    newline=""

) as f:


    writer = csv.writer(
        f
    )


    writer.writerow(

        [
            "epoch",
            "train_mse",
            "train_rel_l2",
            "val_mse",
            "val_mae",
            "val_rel_l2",
            "lr"
        ]
    )


# ============================================================
# Best model path
# ============================================================

best_val = float(
    "inf"
)


best_model_path = os.path.join(

    OUT_DIR,

    "best_galerkin_operator.pt"
)


# ============================================================
# Training loop
# ============================================================

epoch_bar = tqdm(

    range(
        1,
        EPOCHS + 1
    ),

    desc="Training",

    ncols=120
)


for epoch in epoch_bar:


    # ========================================================
    # Train
    # ========================================================

    train_mse, train_rel = train_one_epoch(

        model,

        train_loader,

        optimizer
    )


    # ========================================================
    # Validation
    # ========================================================

    (
        val_mse,
        val_mae,
        val_rel

    ) = evaluate(

        model,

        val_loader
    )


    # ========================================================
    # Scheduler
    # ========================================================

    scheduler.step()


    lr_now = optimizer.param_groups[
        0
    ]["lr"]


    # ========================================================
    # Log
    # ========================================================

    with open(

        log_path,

        "a",

        newline=""

    ) as f:


        writer = csv.writer(
            f
        )


        writer.writerow(

            [
                epoch,
                train_mse,
                train_rel,
                val_mse,
                val_mae,
                val_rel,
                lr_now
            ]
        )


    # ========================================================
    # Save best model
    # ========================================================

    if val_rel < best_val:


        best_val = val_rel


        torch.save(

            model.state_dict(),

            best_model_path
        )


    # ========================================================
    # Print
    # ========================================================

    if (

        epoch == 1

        or

        epoch % 10 == 0
    ):


        print(

            f"\nEpoch {epoch:04d} | "

            f"Train MSE {train_mse:.4e} | "

            f"Train Rel {train_rel:.4e} | "

            f"Val MSE {val_mse:.4e} | "

            f"Val MAE {val_mae:.4e} | "

            f"Val Rel {val_rel:.4e}"
        )


    epoch_bar.set_postfix(

        train_mse=f"{train_mse:.2e}",

        val_mse=f"{val_mse:.2e}",

        rel=f"{val_rel:.2e}",

        lr=f"{lr_now:.1e}"
    )


# ============================================================
# Load best model
# ============================================================

print(
    "\nLoading best model..."
)


model.load_state_dict(

    torch.load(

        best_model_path,

        map_location=DEVICE,

        weights_only=True
    )
)


# ============================================================
# Final test
# ============================================================

(
    test_mse,
    test_mae,
    test_rel

) = evaluate(

    model,

    test_loader
)


print(
    "\nFinal Test Results"
)


print(
    f"Test MSE     : {test_mse:.6e}"
)


print(
    f"Test MAE     : {test_mae:.6e}"
)


print(
    f"Test Rel L2  : {test_rel:.6e}"
)


# ============================================================
# Save results
# ============================================================

with open(

    os.path.join(
        OUT_DIR,
        "test_results.txt"
    ),

    "w"

) as f:


    f.write(
        "Final Test Results\n"
    )


    f.write(

        f"Test MSE     : "
        f"{test_mse:.6e}\n"
    )


    f.write(

        f"Test MAE     : "
        f"{test_mae:.6e}\n"
    )


    f.write(

        f"Test Rel L2  : "
        f"{test_rel:.6e}\n"
    )


# ============================================================
# Plot
# ============================================================

plot_loss(
    log_path
)


plot_prediction(

    model,

    test_dataset,

    sample_index=0
)


# ============================================================
# Finish
# ============================================================

print(

    f"\nBest validation Rel L2: "
    f"{best_val:.6e}"
)


print(

    f"Best model saved to: "
    f"{best_model_path}"
)


print(

    f"\nAll results saved to: "
    f"{OUT_DIR}"
)

Results will be saved to:
galerkin_operator_results/run_20260821_143906
Using device: cuda
Total .mat samples found: 800

Dataset split:
Train samples: 640
Val samples  : 80
Test samples : 80

Calculating normalization statistics...


Statistics:   0%|                                                                               | 0/640 [00:00…


Normalization statistics:
PT mean = 3.135567e+01
PT std  = 2.164740e+01
PA mean = -2.100887e+04
PA std  = 2.847824e+05

Batch check:
PT batch shape: torch.Size([4, 1, 501, 200])
PA batch shape: torch.Size([4, 1, 501, 200])

Checking model dimensions...
Input shape : torch.Size([1, 1, 501, 200])
Output shape: torch.Size([1, 1, 501, 200])

Trainable parameters: 593,377


Training:   0%|                                                                                | 0/1000 [00:00…


Epoch 0001 | Train MSE 9.6197e-01 | Train Rel 1.0987e+00 | Val MSE 8.1740e-01 | Val MAE 2.0685e-01 | Val Rel 9.1173e-01
